# Spatially Variable Genes with Squidpy: Moran's I

Reproducible code for the blog post at [spatiabio.com](https://www.spatiabio.com).

Dataset: 10x Genomics Visium mouse brain demo.

## Setup

In [ ]:
import squidpy as sq
import scanpy as sc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load and preprocess

In [ ]:
adata = sc.read_h5ad("../../data/anndata/visium_hne_adata.h5ad")
# Or load directly: adata = sq.datasets.visium_hne_adata()
print(f"Shape: {adata.shape}")

In [ ]:
# Filter to top 500 HVGs
sc.pp.highly_variable_genes(adata, n_top_genes=500, flavor="cell_ranger")
adata = adata[:, adata.var.highly_variable].copy()
print(f"After HVG filter: {adata.shape}")

# NOTE: visium_hne_adata() ships .X ALREADY log-normalized (max ~8.4, non-integer);
# the raw integer counts live in adata.raw.X. Calling normalize_total + log1p here
# would log it a second time -- silently, with no error, changing every result below.


## Build spatial graph

In [ ]:
sq.gr.spatial_neighbors(adata)
print("Spatial graph built")

## Run Moran's I

> **Windows note**: The `if __name__ == '__main__':` guard is required on Windows due to multiprocessing. On Linux/Mac it is not needed.

In [ ]:
if __name__ == '__main__':
    sq.gr.spatial_autocorr(adata, mode="moran", n_perms=200, n_jobs=1)

    svg_results = adata.uns["moranI"]
    top20 = svg_results.sort_values("I", ascending=False).head(20)
    print("=== Top 20 Spatially Variable Genes ===")
    print(top20[["I", "pval_norm", "pval_norm_fdr_bh"]].to_string())

    svg_results.to_csv("svg_results.csv")
    print("\nSaved to svg_results.csv")

## Results summary

Top SVGs from this run (top 500 HVGs, `n_perms=200`):

| Rank | Gene | Moran's I | Function |
|------|------|-----------|----------|
| 1 | Prkcd | 0.824 | Protein kinase C delta; thalamus-enriched |
| 2 | Pmch | 0.818 | Pro-melanin-concentrating hormone; hypothalamic neuropeptide |
| 3 | Mobp | 0.811 | Myelin-associated oligodendrocyte basic protein; white matter |
| 4 | Agrp | 0.799 | Agouti-related peptide; hypothalamic (arcuate) neuropeptide |
| 5 | Tcf7l2 | 0.794 | Transcription factor; thalamic marker |

> **Corrected.** An earlier version of this notebook reported Itpka (0.674), Fezf1, Baiap3,
> Shox2 and Slc30a3. Those came from a pipeline that called `normalize_total` + `log1p` on
> `visium_hne_adata()`, whose `.X` is **already log-normalized** — so the data was logged
> twice. That silently compressed the values and reshuffled the ranking. The table above is
> from the corrected pipeline; `Itpka` actually lands around rank 11 (I = 0.727).